# Notebook 01: Data Acquisition and Validation

This notebook demonstrates:
1. Fetching stock data using yfinance
2. Caching data locally
3. Data preprocessing and quality validation
4. Generating data quality reports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger

from config.settings import *
from config.universes import get_universe, get_sp500_full
from src.data.fetcher import YFinanceFetcher
from src.data.preprocessor import DataPreprocessor
from src.data.cache_manager import CacheManager

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Configure logger
logger.add('notebook_01.log', rotation='10 MB')

print('✅ Imports successful')

## 1. Initialize Components

In [ ]:
# Initialize fetcher, preprocessor, and cache manager
fetcher = YFinanceFetcher()
preprocessor = DataPreprocessor()
cache = CacheManager()

print('Initialized:')
print(f'  - YFinanceFetcher (rate limit: {YFINANCE_RATE_LIMIT} req/hr)')
print(f'  - DataPreprocessor')
print(f'  - CacheManager (dir: {cache.cache_dir})')

## 2. Define Stock Universe

Start with a small test set, then expand to full universe

In [ ]:
# Option 1: Small test set (5 US + 5 HK stocks)
test_us = ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'NVDA']
test_hk = ['0700.HK', '9988.HK', '1398.HK', '0005.HK', '1299.HK']

# Option 2: Sample universe (from config)
sample_us = get_universe('US', 'sample')[:50]  # First 50
sample_hk = get_universe('HK', 'sample')[:30]  # First 30

# Choose universe for this run
USE_TEST_SET = True  # Set to False for full sample

if USE_TEST_SET:
    us_tickers = test_us
    hk_tickers = test_hk
    print(f'Using TEST set: {len(us_tickers)} US + {len(hk_tickers)} HK stocks')
else:
    us_tickers = sample_us
    hk_tickers = sample_hk
    print(f'Using SAMPLE set: {len(us_tickers)} US + {len(hk_tickers)} HK stocks')

print(f'\nUS tickers: {us_tickers[:5]}...')
print(f'HK tickers: {hk_tickers[:5]}...')

## 3. Fetch US Market Data

In [ ]:
# Date range
start_date = '2020-01-01'
end_date = '2024-12-31'

print(f'Fetching US data from {start_date} to {end_date}...')

# Use cache manager to fetch or load cached data
us_data = cache.get_or_fetch(
    market='US',
    universe='test' if USE_TEST_SET else 'sample',
    tickers=us_tickers,
    start_date=start_date,
    end_date=end_date,
    fetcher=fetcher,
    force_refresh=False
)

print(f'\n✅ US data fetched: {us_data.shape}')
print(f'Columns: {list(us_data.columns)}')
print(f'Date range: {us_data.index.get_level_values("date").min()} to {us_data.index.get_level_values("date").max()}')

## 4. Fetch HK Market Data

In [ ]:
print(f'Fetching HK data from {start_date} to {end_date}...')

hk_data = cache.get_or_fetch(
    market='HK',
    universe='test' if USE_TEST_SET else 'sample',
    tickers=hk_tickers,
    start_date=start_date,
    end_date=end_date,
    fetcher=fetcher,
    force_refresh=False
)

print(f'\n✅ HK data fetched: {hk_data.shape}')
print(f'Date range: {hk_data.index.get_level_values("date").min()} to {hk_data.index.get_level_values("date").max()}')

## 5. Data Preprocessing and Quality Validation

In [ ]:
print('Preprocessing US data...')
us_clean, us_report = preprocessor.validate_data_quality(us_data)

print('\n' + '='*60)
print('US DATA QUALITY REPORT')
print('='*60)
print(f'Initial shape: {us_report["initial_shape"]}')
print(f'Final shape: {us_report["final_shape"]}')
print(f'Initial nulls: {us_report["initial_nulls"]}')
print(f'Final nulls: {us_report["final_nulls"]}')
print(f'Data coverage: {us_report["data_coverage"]:.2%}')
print(f'Tickers count: {us_report["tickers_count"]}')
print(f'Date range: {us_report["date_range"][0]} to {us_report["date_range"][1]}')
if 'removed_tickers' in us_report:
    print(f'Removed tickers: {us_report["removed_tickers"]}')

In [ ]:
print('Preprocessing HK data...')
hk_clean, hk_report = preprocessor.validate_data_quality(hk_data)

print('\n' + '='*60)
print('HK DATA QUALITY REPORT')
print('='*60)
print(f'Initial shape: {hk_report["initial_shape"]}')
print(f'Final shape: {hk_report["final_shape"]}')
print(f'Initial nulls: {hk_report["initial_nulls"]}')
print(f'Final nulls: {hk_report["final_nulls"]}')
print(f'Data coverage: {hk_report["data_coverage"]:.2%}')
print(f'Tickers count: {hk_report["tickers_count"]}')
if 'removed_tickers' in hk_report:
    print(f'Removed tickers: {hk_report["removed_tickers"]}')

## 6. Exploratory Data Analysis

In [ ]:
# Sample a few tickers for visualization
sample_tickers = us_clean.index.get_level_values('ticker').unique()[:3]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for i, ticker in enumerate(sample_tickers):
    ticker_data = us_clean.xs(ticker, level='ticker')
    ticker_data['close'].plot(ax=axes[i], title=f'{ticker} - Adjusted Close Price')
    axes[i].set_ylabel('Price ($)')
    axes[i].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Compute basic statistics
us_stats = us_clean.groupby(level='ticker')['close'].agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max'),
    ('count', 'count')
])

print('\n=== US Stocks Statistics ===')
print(us_stats.head(10))

## 7. Missing Data Analysis

In [ ]:
# Analyze missing data by ticker
missing_by_ticker = us_clean.groupby(level='ticker').apply(
    lambda x: x.isnull().sum().sum() / (len(x) * len(x.columns))
).sort_values(ascending=False)

print('Top 10 tickers with most missing data:')
print(missing_by_ticker.head(10))

# Visualize
plt.figure(figsize=(12, 6))
missing_by_ticker.head(20).plot(kind='bar')
plt.title('Missing Data Ratio by Ticker (Top 20)')
plt.ylabel('Missing Ratio')
plt.xlabel('Ticker')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Cache Information

In [ ]:
cache_info = cache.get_cache_info()

print('\n=== Cache Information ===')
for key, value in cache_info.items():
    print(f'{key}: {value}')

## 9. Save Processed Data

In [ ]:
# Save to processed directory
us_output_path = PROCESSED_DATA_DIR / f'us_clean_{start_date}_{end_date}.parquet'
hk_output_path = PROCESSED_DATA_DIR / f'hk_clean_{start_date}_{end_date}.parquet'

us_clean.to_parquet(us_output_path, compression='snappy')
hk_clean.to_parquet(hk_output_path, compression='snappy')

print(f'\n✅ Saved processed data:')
print(f'  US: {us_output_path}')
print(f'  HK: {hk_output_path}')

## Summary

This notebook demonstrated:
- ✅ Fetching stock data from yfinance with rate limiting
- ✅ Caching data locally to avoid repeated API calls
- ✅ Data preprocessing (adjustment, anomaly detection, missing value handling)
- ✅ Data quality validation and reporting
- ✅ Exploratory data analysis

Next steps:
- Notebook 02: Exploratory analysis of returns and volatility
- Notebook 03: Factor construction (momentum, mean reversion, volatility)